# Specification search and candidate comparisons

`auto_policy` compares a fixed sequence of predictive specifications;
`modeler_loop` repeats a broader but still predetermined diagnostic cycle.
`SearchSession` exposes the same statistical functions as caller-chosen actions.
These routes use
training data to nominate changes and held-out folds within that training set to
choose among fitted candidates. An independent test sample is required to estimate
the performance of the whole chosen procedure. The [machine-output case](../cases/budgeted_interaction_search.ipynb)
shows a budgeted interaction search; the [body-fat case](../cases/bodyfat_modeler.ipynb)
shows domain judgment over measured evidence.

## Terms and limitations

Terms used on this page:

- **incumbent**: the accepted specification that a candidate change is compared against.
- **adoption rule**: a candidate replaces the incumbent only when it lowers held-out risk by more than the stated margin.
- **budget** (`CostBudget`): the cap on native model fits that the workflow may spend across discovery, tuning, scoring, refits and pre-scans. Each fit counted against the cap is *charged* to it.
- **residual tree**: a tree term fitted to what the additive model leaves unexplained, to capture interactions.
- **decision trail**: the record of each stage of the search, the fits used and why each candidate was kept or dropped.
- **agent**: an external caller that chooses the next action in a `SearchSession`.
- **judge**: a callback that reads the digest and proposes edits to the specification; it is either a rule written by a person or an `LLMJudge`.
- **digest**: the summary a judge receives, with the current specification, measured term gains, screened pairs, group and lock declarations, the budget summary and an `evidence_id`.
- **provider**: the external language-model service that an `LLMJudge` calls through a configured chat endpoint, sending it the digest.
- **`SpecificationJudgment`**: the validator that checks the term names and syntax of a judge's proposed edits.
- **`evidence_id`**: an identifier that ties an action or digest to the state it was computed from; a stale identifier is refused.
- **`FitCache`**: a store of completed policy evaluations, keyed by data, fold design, response, family, metric, fitting arguments and build identity.
- **`causal_traps`**: a design check in a causal session that guards proposed candidates.
- **`transform_candidates`**: the native action that fits one preliminary surrogate and returns editable alternatives, such as a `log1p` transform or a `btree` term.
- **`response_probe`**: the comparison of Gaussian identity, `log1p` and `log` responses at one fixed iteration on identical folds.
- **`term_report`**: the action that reports the contribution of each fitted term.
- **`compare`**: the action that fits and scores a complete candidate formula against the current one.
- **`two_se_candidates`**: the summary from `compare` ordered by smallest change in MSE, then term count, then candidate name.
- **`also_cleared`**: the other candidates from `compare` that passed the same rule.
- **`export_transcript()`**: a metadata-only record of a session, saved after the session is frozen.

Limitation: complete trails from `auto_policy`, `modeler_loop` and `SearchSession` have not yet been shown to be identical, because the modeler's initial autospec pass runs before its session.

## Fold score and target

For candidate specification $s$, fold $b$ fits on training rows $T_b$ and scores
predictions on evaluation rows $V_b$. Given a response-scale loss $L$, the
weighted fold score is

$$r_b(s,m)=\frac{\sum_{i\in V_b}v_iL(y_i,\hat\mu^{[m]}_{s,T_b}(x_i))}
                   {\sum_{i\in V_b}v_i},$$

where $m$ is the boosting iteration and $v_i$ is the evaluation weight. The
reported score is the arithmetic mean of fold scores. This gives each fold equal
weight, including when its evaluation-row count differs. For Gaussian mean
prediction with squared-error loss, the unit is the response unit squared.
Pinball and expectile scores also have response-scale interpretations; family
risk can instead have a different unit. Observations omitted from both training
and evaluation do not contribute to the score.

The fold design determines the prediction task. Random folds target new
exchangeable observations. A grouped design holds entire groups out and targets
new groups; if multiple grouping columns are declared, rows connected by *any*
shared group label are held together. Rolling folds target later records under
the supplied ordering. The same design is used when the policy prices its
candidates. A test set reserved before any screening is not read by these steps.

## Stopping iteration and candidate comparison

Each policy candidate has its own stopping iteration,

$$\hat m_s=\operatorname*{arg\,min}_{m\geq1}\frac1B\sum_{b=1}^{B}r_b(s,m).$$

The offset-only model at $m=0$ is excluded from this search. The policy then
compares candidate $c$ with incumbent $a$ on their shared folds using
$d_b=r_b(a,\hat m_a)-r_b(c,\hat m_c)$. A positive $d_b$ favours $c$. For one
partition of $B$ folds, its mean gain and descriptive standard error are

$$\bar d=\frac1B\sum_b d_b,\qquad
  \operatorname{SE}(\bar d)=
  \sqrt{\frac{1}{B(B-1)}\sum_b(d_b-\bar d)^2}.$$

The balanced policy adopts $c$ only when $\bar d>2\operatorname{SE}(\bar d)$;
`complexity="rich"` uses one standard error. The comparison is computed in
`mboostrs-core/src/selection.rs`. Repeated partitions reuse observations:
the implementation computes variance within each partition and averages those
variances before dividing by folds per partition; stacked fold losses are therefore not treated as independent.

For example, if the incumbent's losses on three folds are 1.20, 1.04 and 1.10
response units², and the candidate's are 1.12, 1.06 and 1.02, the paired gains
are 0.08, -0.02 and 0.08. Their mean is 0.0467 response units² and their
standard error is 0.0333 response units². The gain clears one standard error
but not two; a balanced policy keeps the incumbent. This is a calculation
about consistency across these three folds. Its standard error describes the spread of the paired gains and does not state an error rate.

Stopping points, shapes and proposed terms were all selected after examining
these folds. The fold difference therefore understates the uncertainty of an
external performance claim. The final model is fitted on all training rows at
its selected iteration, and the separate test sample evaluates that complete
training procedure once. If a candidate fails the adoption rule, its terms are
not carried forward, but later stages may test different structure against
the same retained incumbent.

## Policy stages and profiles

The first policy stage prices a linear or group-effect specification. Later
stages test adaptive main-effect shapes, nominated pair terms and a residual
tree. The main-effect stage runs `autospec` within its `discovery_fits`
allowance. Its boosted-tree, neural or dual pre-scan supplies additional pair
leads. The pair stage also screens response-scale residuals of the *accepted*
full-data model at its selected iteration; it does not fit a deeper surrogate
when that iteration is below 50. It joins valid preliminary leads and tests
the combined incumbent-plus-pair formula on the shared folds. The tree stage
tests one depth-5 tree over the accepted model's eligible columns. It retains
accepted main effects and pairs and does not repeat additive discovery. These
leads are candidates for a *complete model* comparison, not evidence that
they improve prediction.

The profile is `linear` for the initial candidate, `additive` for changed main
effects, `tensor` for an accepted `%X%` interaction, `varying` for accepted
`by=` terms without a tensor, and `hybrid` when the returned predictor includes
a residual tree. These labels describe the structure of the returned *predictor* only. A dual tree/neural pre-scan, for example, can still return an additive or varying model.

The requested score and the family risk can differ. The main-effect shape catalogue ranks candidates by family risk, and the residual pair screen ranks them by response-scale squared-error removal, so the two nomination steps use different scales. The policy tunes and adopts complete candidates
under the requested score. A strong candidate missed by either screen cannot
be recovered by the later comparison. These limits belong to the search
procedure; an independent test sample checks its total performance.

## Agent-directed search: `SearchSession`

A `SearchSession` fixes the training data, fold design and scoring loss before fitting. It exposes
one legal next operation at a time, each with a stated purpose, minimum fit
cost and result status. An external agent may choose `column_screen` when the
variable roles are uncertain, `term_report` when a fitted term may be
redundant, `interaction_screen` when residual joint structure remains, or
`compare` when a domain alternative is ready to test. The caller chooses these actions in response to the current results.
`final_evaluate` scores the reserved test sample only after the specification
has been frozen.

A causal `SearchSession` additionally locks its declared baseline covariates,
excludes treatment and post-treatment columns from candidate screens, and
exposes `causal_traps` as an explicit design check. This guards proposals;
it does not identify an effect.

An action's evidence identifier chains it to the preceding state. A JSON
checkpoint retains hypotheses, comparisons, fold design and cumulative native
fit charges. Resume checks the training fingerprint, native build and chain;
it does not serialize a fitted native model, so a fresh fit costs another
charge when needed. A candidate can be adopted only from a measured
comparison against the current specification and under the stated paired
fold rule. A hypothesis or pre-scan is recorded as screening evidence, whose role is to rank candidates for the search to test. Test-set scoring is one-time by this interface,
although a caller outside the interface must still keep those rows private.

The `SearchSession` layer orchestrates existing Python public functions;
all fitting and response scoring stay in the Rust core. An optional
`SearchSession.drive(judgment, max_actions=...)` selects from the same available actions
within the Python process and logs provider call hashes when given an
`LLMJudge`. It never submits the final test evaluation. `auto_policy` and
`modeler_loop` still choose fixed action sequences. Their measured fits,
screens, comparisons and tuning use shared actions and expose the resulting
evidence. The budgeted policy also records its initial fallback fit and
pair/tree nominations. The modeler's initial autospec pass precedes its session. The [commuting-zone case](../cases/cz_agent_search.ipynb)
demonstrates a provider-free agent-directed investigation and why a selected
variable's role cannot be read from its score alone.

## Screening nominations and interpretable alternatives

The native `transform_candidates` action fits one preliminary surrogate in
native mode and returns editable alternatives. A right-skewed nonnegative
predictor may be refitted on `log1p(x)`; a step-like pair may be represented by
`btree(z, by=x, maxdepth=1)`, a varying slope with one learned split. The
surrogate's shape and Friedman $H^2$ statistics nominate the pair. The
`interaction_screen` quantity called `removed` is residual MSE removed on a
held-out half-sample. It is a screening number; a nested `compare` supplies the fold-paired standard error on which adoption of a complete candidate is decided. A weak term highly correlated with a locked numeric sibling may
receive a proxy note from `term_report(proxy_hygiene=True)`; the lock remains
in place and the analyst must compare the proposed nested drop.

`compare` retains the input order of candidate rows. Its
`two_se_candidates` summary is separately ordered by smallest change in
MSE, then term count and candidate name. `also_cleared` preserves the other
candidates that passed; a caller may choose an interpretable tensor rather
than the top-ranked tree. No `adopt` call is automatic. A smaller nested
formula can be kept despite an error increase only when the caller supplies
`simplification_se=`; the default zero tolerates no increase, and two allows
an increase within two paired fold standard errors. These are descriptive search preferences: they set the increase in fold error that the search accepts for a smaller formula.

For a nonnegative or positive response, `response_probe` can compare Gaussian
identity, log1p and log fits at one fixed iteration on identical folds. The
inverse transform and MSE are computed on the original response scale by the
Rust core. Exponentiating a fitted mean log response gives a back-transformed
point prediction, the conditional geometric mean of the response on the original scale; it differs from the conditional mean, and that difference matters when interpreting its MSE. The probe changes neither
the session's family nor its accepted formula. A residual-spread warning
points to a separately declared location-scale family rather than silently
changing the target. Every candidate fit is charged to the same session
budget. A metadata-only `export_transcript()` can be saved after freeze;
its evidence and final score can be audited but cannot resume fitting.

## Judgment cycle

`modeler_loop` starts with `autospec` discovery and repeats the following
operations for at most `max_rounds`: fit the current specification, summarize
term gains and residual pair leads, pass an aggregate digest to a judge,
validate its proposed edits, compare eligible complete specifications, and
retain the edit supported by the fold rule. It ends with a final stopping-
iteration sweep. The public `ModelerResult` contains the fitted model,
editable specification, transcript, refusals and fit accounting. This loop
compares family risk; it does not inherit `auto_policy`'s requested
response-score interface or its explicit fold design.

The digest contains the current specification, measured term gains, screened
pairs, group and lock declarations, budget summary and an `evidence_id`.
A judge can propose dropping, adding or replacing a term, locking a predictor,
or constraining a monotone effect. `SpecificationJudgment` validates the proposed term
names and syntax; a stale evidence identifier or a proposed edit that removes
a locked predictor is refused. A valid proposal enters the same measured
comparison as deterministic suggestions. A reduction in model size may be
retained when its fold error increase does not reach two paired standard
errors; this is a stated search preference: it sets the tolerance for accepting a smaller model.

A judgment callback can be a human-authored rule or an `LLMJudge` using a
configured chat endpoint. The provider receives the digest and supplied
domain context, not the reserved test rows. Proposed edits must pass term validation and the paired fold comparison
before they can change the fitted model.
The transcript connects evidence, proposal and comparison identifiers.
Provider call records store model and endpoint identity, response/prompt
hashes and reported token usage when available. The token limit is checked
between requests; it is not a hard limit on the last provider response.
These records establish *what was proposed and measured*, not the quality
or truth of the provider's reasoning.

## Workflow budget and fallback

A `CostBudget` limits native fit admissions across discovery, tuning, scoring,
refits and pre-scans. It differs from an autospec probe allowance: the latter
only determines how many discovery fits may be proposed at one stage. A fit
limit is checked before native calls. A CV batch reserves its fold charges
before dispatch, and a failed batch conservatively keeps that reservation
because the binding cannot report how many folds completed. Time and memory
limits are cooperative checks between calls. Memory is the process's peak resident set size in MiB over its lifetime, so it includes memory used before this search began.

The workflow first fits a simple candidate. If exhaustion occurs before a
validation comparison completes, the retained model can predict but its
`score` is `None` because no validation comparison completed. If a scored incumbent
exists, exhaustion returns it with the completed decision trail. If even the
first fit cannot be admitted, a structured `workflow_budget` error is raised.
The `FitCache` keys a completed policy evaluation by data, fold design,
response, family, metric, fitting arguments and build identity. A cache hit
returns the stored value without charging its historical fits. Prepared
designs are not shared across different formulas.

## Final evaluation and causal assumptions

The search produces a specification and a predictive model, not a causal
estimand or a confidence interval valid after selection. For a causal question,
state treatment timing, adjustment assumptions and an estimand before entering
the causal lane; see [](../usage/causal.ipynb). For a predictive claim, freeze
the procedure on development data, reserve evaluation rows before every
adaptive step and report complete-run failures alongside accuracy. The two
case studies linked in the introduction show independent prediction checks for the implemented
policy and judgment loop; the live LLM extension remains an opt-in example,
not a provider-backed performance study.